# Jev 决策模型

## 特点

Jev 是 TypeSafe 出品的决策模型，它可以低延迟低成本的处理各种决策问题，比如：

- 意图识别
- 行为预测
- 评估结果
- ...

<img src="assets/llm-vs-jev-output.png" width="80%">

## 配置 LiteLLM 网关

1. 覆盖 `docker-compose.yml` 文件到你的工程
2. 重建容器：`docker compose up -d --no-deps --force-recreate litellm`
3. 访问 `http://localhost:4000/ui/`，注意不使用缓存
4. 添加百炼模型
   1. 提供商：`TypeSafe`
   2. 模型ID：`decision-model-preview`
   3. 分组名：`decision`
   4. API Base：`https://llm-bxsnzgjeej076kci.cn-beijing.maas.aliyuncs.com/compatible-mode`
   5. API Key：百炼的API Key
   6. 测试类型：`decision`

## 原生 HTTP 调用

In [ ]:
# 准备问题

# 交代问题背景
state = "同一笔订单被扣了两次款，用户要求人工退款。"

# 问题列表（字典形式）
questions = {
    # 键为自定义的值，表达问题的ID，值是问题的详细信息
    "department": {
        "type": "choice",   # 选择题
        "instructions": "这张工单应交给哪个部门？", # 问题主体
        "criteria": {   # 选项
            "billing": "支付、退款、重复扣款和账单问题",
            "technical": "产品故障、登录失败和接口集成问题",
            "other": "不属于以上类别，或信息不足以确定部门",
        },
    },
    "needs_human": {
        "type": "noul",     # 判断题
        "instructions": "是否需要人工客服继续处理？",   
        "criteria": {
            "true": "用户要求人工介入，或存在尚未处理的扣款、退款争议",
            "false": "仅查询一般知识，且无需人工操作或处理争议",
        },
    },
    "severity": {
        "type": "score",    # 评分题
        "instructions": "按工单已经提供的信息判断问题严重程度，不猜测额外损失。",
        "criteria": [   # 关键点的分值：0、1、2、3
            "一般咨询，无功能或资金影响",
            "部分使用受影响，但存在替代方案",
            "核心功能不可用，或存在尚未处理的资金争议",
            "已经明确造成大范围业务中断或重大资金损失",
        ],
    },
}

In [ ]:
import httpx
from deal_pilot.config.openai_settings import openai_settings

async with httpx.AsyncClient(timeout=60) as client:
    response = await client.post(
        "http://localhost:4000/v1/systemone",
        headers={"Authorization": f"Bearer {openai_settings.api_key}"},
        json={
            "model": "decision",
            "state": state,
            "questions": questions,
        },
    )
    # 保留 HTTP 错误，不把鉴权失败、限流等情况伪装成模型判断。
    response.raise_for_status()
    result = response.json()

In [ ]:
from rich import print
print(result)

## 使用 TypeSafe SDK

In [ ]:
# !uv add typesafe-sdk
!uv sync

In [6]:
from typesafe_sdk import AsyncTypeSafeClient, Choice, Noul, Score
from deal_pilot.config.openai_settings import openai_settings

async with AsyncTypeSafeClient(
    api_key=openai_settings.api_key,  # 网关的 Key
    base_url=openai_settings.root_url,      # 网关根地址
    model="decision",                    # 你配置的模型名称
) as client:
    result = await client.system_one(
        state="同一笔订单被扣了两次款，用户要求人工退款。",
        questions={
            "department": Choice(
                instructions="应该交给哪个部门？",
                criteria={
                    "billing": "支付、账单和退款问题",
                    "technical": "技术故障",
                    "other": "其他问题",
                },
            ),
            "needs_human": Noul(
                instructions="是否需要人工客服处理？",
            ),
            "severity": Score(
                instructions="问题有多严重？",
                criteria=[
                    "一般咨询，无功能或资金影响",
                    "部分使用受影响，但存在替代方案",
                    "核心功能不可用，或存在资金争议",
                    "大范围业务中断或重大资金损失",
                ],
            ),
        },
    )

In [7]:
from rich import print
print(result)

SystemOneResponse(
    model='decision-model-preview',
    usage=Usage(input_tokens=112, output_tokens=0),
    answers={
        'department': ChoiceAnswer(
            type='choice',
            choice='billing',
            confidence=0.45,
            probabilities={'billing': 0.63, 'technical': 0.0, 'other': 0.37}
        ),
        'needs_human': NoulAnswer(type='noul', noul=0.99),
        'severity': ScoreAnswer(
            type='score',
            score=1.66,
            confidence=0.88,
            legend={
                0: '一般咨询，无功能或资金影响',
                1: '部分使用受影响，但存在替代方案',
                2: '核心功能不可用，或存在资金争议',
                3: '大范围业务中断或重大资金损失'
            },
            probabilities={0: 0.13, 1: 0.1, 2: 0.76, 3: 0.01}
        )
    }
)

In [10]:
print(result.choices)

{
    'department': ChoiceAnswer(
        type='choice',
        choice='billing',
        confidence=0.45,
        probabilities={'billing': 0.63, 'technical': 0.0, 'other': 0.37}
    )
}